In [41]:
import pandas as pd
import joblib
import sqlite3

from sklearn.ensemble import RandomForestRegressor

In [43]:
data = pd.read_csv(r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\final_feature_set.csv")

data.head()

,job_id,person_id,title,required_experience,total_skills,match_relevance_score
0,15796,20608,Beauty & Fragrance consultants needed,NaN,19,0.5
1,861,37600,Retail Territory Merchandiser,NaN,113,0.9
2,5391,51921,Inside Sales Rep,Associate,66,0.9
3,11965,28530,Cities Project Manager,NaN,26,0.5
4,11285,35955,Digital Procurement Assistant,Entry level,58,0.9


In [45]:
X = data[["total_skills"]]
y = data["match_relevance_score"]

model = RandomForestRegressor(random_state=42)
model.fit(X, y)

print("Model Trained Successfully!")

Model Trained Successfully!


In [47]:
joblib.dump(model,
            r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\recruitsmart_model.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [49]:
loaded_model = joblib.load(
    r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\recruitsmart_model.pkl"
)

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [51]:
connection = sqlite3.connect(
    r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\recruitsmart.db"
)

In [55]:
import os

db_path = r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\recruitsmart.db"

print(os.path.exists(db_path))
print(db_path)

True
C:\Users\cbala\OneDrive\Desktop\RecruitSmart\recruitsmart.db


In [57]:
import os

db_path = r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\recruitsmart.db"

print("Size:", os.path.getsize(db_path), "bytes")

Size: 96567296 bytes


In [59]:
import pandas as pd
import sqlite3

connection = sqlite3.connect(
    r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\recruitsmart.db"
)

In [61]:
jobs = pd.read_csv(
    r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\data\jobs_cleaned.csv"
)

people = pd.read_csv(
    r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\data\01_people.csv"
)

applications = pd.read_csv(
    r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\data\applications.csv"
)

In [62]:
jobs.to_sql("Jobs", connection, if_exists="replace", index=False)

people.to_sql("Candidates", connection, if_exists="replace", index=False)

applications.to_sql("Applications", connection, if_exists="replace", index=False)

connection.commit()

print("Database Created Successfully!")

Database Created Successfully!


In [63]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", connection)

,name
0,Jobs
1,Candidates
2,Applications


In [64]:
import os

print(os.path.getsize(
    r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\recruitsmart.db"
))

96567296


In [66]:
query = """
SELECT
    job_id,
    title,
    required_experience
FROM Jobs
LIMIT 10;
"""

jobs = pd.read_sql(query, connection)

jobs.head()

,job_id,title,required_experience
0,1,Marketing Intern,Internship
1,2,Customer Service - Cloud Video Production,Not Applicable
2,3,Commissioning Machinery Assistant (CMA),None
3,4,Account Executive - Washington DC,Mid-Senior level
4,5,Bill Review Manager,Mid-Senior level


In [71]:
from fastapi import FastAPI

In [74]:
app = FastAPI()

print("FastAPI App Created Successfully!")

FastAPI App Created Successfully!


In [76]:
@app.get("/jobs/{job_id}")
def get_job(job_id: int):

    query = f"""
    SELECT *
    FROM Jobs
    WHERE job_id = {job_id}
    """

    result = pd.read_sql(query, connection)

    return result.to_dict(orient="records")

In [78]:
get_job(5)

[{'job_id': 5,
  'title': 'Bill Review Manager',
  'location': 'US, FL, Fort Worth',
  'department': None,
  'salary_range': None,
  'company_profile': 'SpotSource Solutions LLC is a Global Human Capital Management Consulting firm headquartered in Miami, Florida. Founded in January 2012, SpotSource has created a fusion of innovative service offerings to meet the increasing demand of today’s economy. We specialize in Talent Acquisition, Staffing, and Executive Search Services across various functions and in specific industries. Global Talent Transfusion (GTT) services utilize best in practice qualification standards to deliver talent in temporary, temporary-to-hire, and permanent basis. Health Career Transition (HCT) is a subsidiary of Global Talent Transfusion and offers placement services specifically in the growing Healthcare arena. SpotSource Executive Search (SES) Consultants are special breed talent evangelists that understand how to advise and streamline the human resources proce

In [80]:
from fastapi import FastAPI
import pandas as pd
import sqlite3

app = FastAPI()

connection = sqlite3.connect(
    r"C:\Users\cbala\OneDrive\Desktop\RecruitSmart\recruitsmart.db",
    check_same_thread=False
)

@app.get("/jobs/{job_id}")
def get_job(job_id: int):

    query = f"""
    SELECT *
    FROM Jobs
    WHERE job_id={job_id}
    """

    result = pd.read_sql(query, connection)

    return result.to_dict(orient="records")

In [82]:
!pip show uvicorn

Name: uvicorn
Version: 0.37.0
Summary: The lightning-fast ASGI server.
Home-page: https://uvicorn.dev/
Author: 
Author-email: Tom Christie <tom@tomchristie.com>, Marcelo Trylesinski <marcelotryle@gmail.com>
License: 
Location: C:\Users\cbala\anaconda3\Lib\site-packages
Requires: click, h11
Required-by: gradio
